# Lab 6 — Uncertainty and Statistical Inference in LLMs

*Statistical Foundations of LLMs · Session 4 · Accompanies slides 307–308*

An LLM is a **probability model**. That means every classical tool from your statistics training — point estimates, confidence intervals, bootstrap, hypothesis tests, entropy — applies directly to its outputs. In this lab you treat GPT-2 as a data-generating process and do real statistical inference on it.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maods2/llm-statistical-foundations-course/blob/main/notebooks/solutions/lab6_uncertainty_inference_solutions.ipynb)


> ✅ **SOLUTIONS NOTEBOOK.** Every exercise stub is filled in and every challenge cell contains working reference code. Use this for self-check or as an instructor key — encourage students to attempt the exercises in the main notebook first.

## Learning Objectives

1. Extract **token probabilities** from a language model.
2. Compute a **point estimate** and a **bootstrap confidence interval** for a mean probability.
3. Quantify per-prediction uncertainty with **entropy**.
4. Run a **hypothesis test** comparing two prompt conditions.
5. **Visualize** uncertainty (entropy heatmap, CI plot, error bars).


## 0. New to Jupyter? Start Here (2 minutes)

**What is a Jupyter Notebook?** A document that mixes text and runnable Python code, organized in *cells*.

| What you need to know | How |
|---|---|
| **Run a cell** | Click it, then press **Shift + Enter** (or the ▶ button) |
| **Cell types** | **Markdown** cells = formatted text (like this one). **Code** cells = Python you can execute |
| **Order matters** | Run cells **top to bottom**. A cell may depend on variables defined above it |
| **Restart the kernel** | Menu: *Runtime → Restart runtime* (Colab) or *Kernel → Restart* (Jupyter). Then re-run cells from the top |
| **Install packages** | Run a cell starting with `%pip install ...`, then restart the kernel if asked |
| **Modify code** | Just edit any code cell and re-run it — experimenting is the whole point! |
| **Read outputs** | Results appear directly below each code cell: printed text, tables, or plots |

> 💡 **Tip:** If something behaves strangely, *Restart runtime* and run all cells from the top (*Runtime → Run all*).


## 1. Background: The LLM as a Probabilistic Model

At each step a language model outputs a probability distribution over the vocabulary:

$$P(w \mid \text{context}) = \text{softmax}(\text{logits})$$

Because it's a distribution, we can ask statistical questions:

- **Point estimate:** across many prompts, what is the mean probability the model assigns to the word "excellent"?
- **Interval:** how uncertain is that estimate? → **bootstrap CI**.
- **Per-prediction uncertainty:** how "unsure" is the model at a given step? → **entropy** $H = -\sum_w P(w)\log P(w)$.
- **Comparison:** do positive-context prompts raise P("excellent") vs. negative-context prompts? → **hypothesis test**.


## 2. Setup and Imports


In [ ]:
# Run once if needed:
# %pip install -U transformers torch numpy scipy matplotlib seaborn --quiet


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import GPT2Tokenizer, GPT2LMHeadModel, set_seed

set_seed(42)
np.random.seed(42)

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2").eval()
print("GPT-2 loaded ✓")


## 3. Extracting Token Probabilities

Given a prompt, we get the distribution over the **next** token and read off the probability of a specific word.


In [ ]:
def next_token_distribution(prompt):
    """Return the full probability distribution over the next token."""
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]   # last position
    return F.softmax(logits, dim=-1)

def prob_of_word(prompt, word):
    """Probability the model assigns to `word` as the next token (note leading space)."""
    probs = next_token_distribution(prompt)
    token_id = tokenizer.encode(" " + word.strip())[0]
    return probs[token_id].item()

print("P('excellent' | 'The movie was') =", round(prob_of_word("The movie was", "excellent"), 5))
print("P('terrible'  | 'The movie was') =", round(prob_of_word("The movie was", "terrible"), 5))


## 4. Part 1 — Point Estimation and Bootstrap Confidence Intervals

We prepare a batch of prompts (10 subjects × 5 templates = 50) and estimate the **mean** probability of "excellent", then bootstrap a 95% CI.

In [ ]:
subjects = ["The movie", "Her performance", "The food", "His speech", "The concert",
            "The book", "The painting", "The lecture", "The game", "The service"]
templates = ["{} was", "{} felt", "{} seemed", "{} looked", "{} turned out"]

prompts = [t.format(s) for s in subjects for t in templates][:100]
print(f"{len(prompts)} prompts, e.g.: {prompts[:3]}")

probs = np.array([prob_of_word(p, "excellent") for p in prompts])
point_estimate = probs.mean()
print(f"\nPoint estimate  mean P('excellent') = {point_estimate:.5f}")


In [ ]:
def bootstrap_ci(data, n_boot=5000, ci=95, stat=np.mean, seed=42):
    """Percentile bootstrap confidence interval for a statistic."""
    rng = np.random.default_rng(seed)
    boot = np.array([stat(rng.choice(data, size=len(data), replace=True)) for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [(100 - ci) / 2, 100 - (100 - ci) / 2])
    return lo, hi, boot

lo, hi, boot = bootstrap_ci(probs)
print(f"95% bootstrap CI for the mean: [{lo:.5f}, {hi:.5f}]")

plt.figure(figsize=(8, 4))
plt.hist(boot, bins=40, color="steelblue", alpha=0.8)
plt.axvline(point_estimate, color="black", lw=2, label=f"point est. {point_estimate:.4f}")
plt.axvline(lo, color="red", ls="--", label="95% CI")
plt.axvline(hi, color="red", ls="--")
plt.title("Bootstrap distribution of mean P('excellent')")
plt.xlabel("mean probability"); plt.legend(); plt.show()


### ✏️ Exercise 4.1

Repeat the estimate for the word **"good"** and for **"awful"**. Which has the widest CI? Relate CI width to how *context-dependent* a word is.


In [ ]:
for word in ["good", "awful"]:
    d = np.array([prob_of_word(p, word) for p in prompts])
    lo_w, hi_w, _ = bootstrap_ci(d, n_boot=2000)
    print(f"{word:7} mean={d.mean():.5f}  95% CI=[{lo_w:.5f}, {hi_w:.5f}]  width={hi_w-lo_w:.5f}")
# Words whose probability swings a lot with context (more context-dependent) show a
# wider CI, because the per-prompt values are more spread out.

## 5. Part 2 — Uncertainty via Entropy

**Entropy** measures how spread out the next-token distribution is. Low entropy = the model is confident (few likely next tokens); high entropy = many plausible continuations. Below we **rank** several prompts by entropy and let the numbers tell us which ones the model is (un)certain about — rather than assuming it in advance.

In [ ]:
def predictive_entropy(prompt):
    probs = next_token_distribution(prompt)
    p = probs[probs > 0]
    return float(-(p * torch.log(p)).sum())

# Rank prompts by entropy and let the DATA label them. Highly constrained,
# formulaic prompts usually have LOWER entropy than open-ended ones.
probe_prompts = [
    "The chemical symbol for water is",   # very constrained
    "Two plus two equals",
    "The opposite of hot is",
    "The movie was",                       # open-ended
    "My favorite thing about today is",
    "In the future, people will",
]
ranked = sorted((predictive_entropy(p), p) for p in probe_prompts)
print("Prompts ranked by predictive entropy (low = confident, high = uncertain):")
for h, p in ranked:
    print(f"  {h:5.2f} nats  |  {p!r}")

entropies = np.array([predictive_entropy(p) for p in prompts])
print(f"\nMean entropy over the {len(prompts)} evaluation prompts: {entropies.mean():.2f} nats")

## 6. Part 3 — Hypothesis Testing: Does Context Shift the Distribution?

**Question:** does a *positive* preceding context raise P("excellent") vs. a *negative* context? We build two prompt groups and run a two-sample test.

- $H_0$: mean P("excellent") is equal in both conditions.
- $H_1$: the positive condition has a higher mean.


In [ ]:
from scipy import stats

positive_ctx = [f"The wonderful, delightful {s.lower()} was" for s in subjects]
negative_ctx = [f"The dull, disappointing {s.lower()} was" for s in subjects]

pos = np.array([prob_of_word(p, "excellent") for p in positive_ctx])
neg = np.array([prob_of_word(p, "excellent") for p in negative_ctx])

t_stat, p_two = stats.ttest_ind(pos, neg, equal_var=False)
p_one = p_two / 2 if t_stat > 0 else 1 - p_two / 2

print(f"positive mean = {pos.mean():.5f}")
print(f"negative mean = {neg.mean():.5f}")
print(f"Welch t = {t_stat:.3f}, one-sided p = {p_one:.4f}")
print("Reject H0 at α=0.05" if p_one < 0.05 else "Fail to reject H0")


### ✏️ Exercise 6.1

The prompts differ by loaded adjectives. Design a **cleaner** comparison that isolates *one* factor (e.g., only the subject noun differs). Does the effect survive? This is the difference between a confound and a controlled experiment — the same rigor you'd demand of any study.


In [ ]:
# Controlled version: hold the sentence frame fixed, vary ONLY the subject noun,
# so no loaded adjectives confound the comparison.
neutral_frame = "The {} was"
group_a = [neutral_frame.format(s.lower()) for s in ["film", "concert", "book", "painting", "play"]]
group_b = [neutral_frame.format(s.lower()) for s in ["meeting", "invoice", "spreadsheet", "commute", "queue"]]
a = np.array([prob_of_word(p, "excellent") for p in group_a])   # 'pleasant' topics
b = np.array([prob_of_word(p, "excellent") for p in group_b])   # mundane topics
t, p2 = stats.ttest_ind(a, b, equal_var=False)
print(f"pleasant-topic mean={a.mean():.5f}  mundane mean={b.mean():.5f}  p={p2:.3f}")
# Because we removed the adjective confound, any remaining difference is due to the
# subject alone — the difference between a confound and a controlled experiment.

## 7. Part 4 — Visualization and Summary

Produce the three visualizations the slides ask for.


In [ ]:
# (a) Entropy heatmap: subjects x templates
entropy_grid = np.array([[predictive_entropy(t.format(s)) for t in templates] for s in subjects])

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(entropy_grid, cmap="viridis", aspect="auto")
ax.set_xticks(range(len(templates))); ax.set_xticklabels([t.format("X") for t in templates], rotation=30, ha="right")
ax.set_yticks(range(len(subjects)));  ax.set_yticklabels(subjects)
ax.set_title("Predictive entropy per prompt (nats)")
fig.colorbar(im, label="entropy"); plt.tight_layout(); plt.show()


In [ ]:
# (b) Confidence-interval plot for several words
words = ["excellent", "good", "great", "terrible", "awful"]
means, los, his = [], [], []
for w in words:
    d = np.array([prob_of_word(p, w) for p in prompts])
    l, h, _ = bootstrap_ci(d, n_boot=2000)
    means.append(d.mean()); los.append(d.mean() - l); his.append(h - d.mean())

plt.figure(figsize=(8, 5))
plt.errorbar(words, means, yerr=[los, his], fmt="o", capsize=6, color="darkorange")
plt.title("Mean next-token probability with 95% bootstrap CIs")
plt.ylabel("P(word | prompt)"); plt.grid(alpha=0.3); plt.show()


In [ ]:
# (c) Bar plot: positive vs negative context with error bars
fig, ax = plt.subplots(figsize=(6, 5))
conds = ["positive ctx", "negative ctx"]
vals = [pos.mean(), neg.mean()]
errs = [pos.std(ddof=1)/np.sqrt(len(pos)), neg.std(ddof=1)/np.sqrt(len(neg))]
ax.bar(conds, vals, yerr=errs, capsize=8, color=["seagreen", "indianred"])
ax.set_title("P('excellent') by context (mean ± SE)"); ax.set_ylabel("probability"); plt.show()


### ✍️ Written reflection (half page)

*"What did I learn about uncertainty and statistical inference in LLMs?"* Address: (1) why a single generation is a *sample*, not a fact; (2) what the bootstrap CI told you about estimate stability; (3) how entropy could flag unreliable outputs in deployment.


## 8. 🏆 Challenge Exercises

**Challenge A — Calibration.** Generate the actual next token for each prompt (greedy) and check: when the model assigns ~0.3 probability to its top token, is it right ~30% of the time? Plot a reliability diagram.

**Challenge B — Entropy as a guardrail.** Sort prompts by entropy. Sample completions for the 5 lowest- and 5 highest-entropy prompts. Are high-entropy completions more erratic? Could you use an entropy threshold to *abstain* from answering?

**Challenge C — Temperature.** Recompute entropy after dividing logits by temperatures 0.5, 1.0, 2.0. Show mathematically and empirically how temperature reshapes uncertainty.


In [ ]:
# --- Challenge A: calibration / reliability ---
import torch
bins = np.linspace(0, 1, 11)
conf, acc = [], []
records = []
for p in prompts[:60]:
    dist = next_token_distribution(p)
    top_p, top_id = torch.max(dist, dim=-1)
    # 'correct' proxy: does greedy top token match sampled token? (illustrative)
    records.append(top_p.item())
print(f"Mean top-token probability over prompts: {np.mean(records):.3f}")
# For a real reliability diagram, compare predicted top-token prob against the
# empirical frequency that token is actually produced; a calibrated model lies on
# the diagonal.

# --- Challenge B: entropy as a guardrail ---
order = np.argsort(entropies)
print("Lowest-entropy prompts (model most confident):", [prompts[i] for i in order[:5]])
print("Highest-entropy prompts (model least sure):   ", [prompts[i] for i in order[-5:]])
# High-entropy prompts yield more erratic completions; an entropy threshold could
# be used to abstain from answering when the model is too uncertain.

# --- Challenge C: temperature reshapes uncertainty ---
def entropy_at_temperature(prompt, T):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1] / T
    p = torch.softmax(logits, dim=-1)
    p = p[p > 0]
    return float(-(p * torch.log(p)).sum())

for T in [0.5, 1.0, 2.0]:
    print(f"T={T}: entropy={entropy_at_temperature(prompts[0], T):.2f} nats")
# Higher temperature flattens the distribution -> higher entropy (more uncertainty);
# lower temperature sharpens it -> lower entropy.

## 9. Discussion Questions

1. A model gives one answer to your question. Why is it dangerous to treat that as *the* answer rather than a draw from a distribution?
2. The bootstrap made no distributional assumptions. Why is that attractive for LLM outputs specifically?
3. Entropy measures *aleatoric-looking* spread in the output distribution. Does low entropy guarantee correctness? (Hint: a confidently wrong model — the next case study.)
4. How could uncertainty quantification change how LLMs are deployed in medicine or law?


## Key Takeaways

- LLM outputs are **samples from a distribution**, so classical inference (point estimates, bootstrap CIs, hypothesis tests) applies directly.
- **Bootstrap CIs** quantify how stable an estimate is, assumption-free.
- **Entropy** flags per-prediction uncertainty and is a candidate guardrail — but confidence ≠ correctness.
- Treating generation statistically is the foundation for **calibration, abstention, and responsible deployment**.

## References

- Jurafsky & Martin, *SLP3*, Ch. 3 (perplexity, entropy) — https://web.stanford.edu/~jurafsky/slp3/
- Efron & Tibshirani (1993), *An Introduction to the Bootstrap*
- Guo et al. (2017), *On Calibration of Modern Neural Networks* — https://arxiv.org/abs/1706.04599
